<a href="https://colab.research.google.com/github/GOKUL5287/AQI-Air-_pollution_prediction/blob/Kanchana/notebooks/kanchana/D1_build_station_hour_merged.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# D1: Build `station_hour_merged.parquet`

Owner: Kanchana. Reviewer: Gokul.

**Input** (shared Drive `raw/`): `station_hour.csv`, `stations.csv`, `openmeteo_india_weather.csv` (with `Wind_Direction_deg`).
**Output** (shared Drive): `processed/station_hour_merged.parquet` (24 columns) and `samples/KL007_sample.csv`.

## 1. Setup and input check

In [ ]:
import os
from pathlib import Path

import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

# The shared folder must appear in My Drive as a shortcut named exactly "AQI_Project_Data"
DATA_DIR = Path(os.environ.get("AQI_DATA_DIR", "/content/drive/MyDrive/AQI_Project_Data"))
DATA_RAW = DATA_DIR / "raw"
DATA_PROCESSED = DATA_DIR / "processed"
DATA_SAMPLES = DATA_DIR / "samples"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
DATA_SAMPLES.mkdir(parents=True, exist_ok=True)

for name in ["station_hour.csv", "stations.csv", "openmeteo_india_weather.csv"]:
    print(f"{name:32s} found: {(DATA_RAW / name).exists()}")

assert all((DATA_RAW / n).exists() for n in
           ["station_hour.csv", "stations.csv", "openmeteo_india_weather.csv"]), \
    "A raw file is missing. Check the Drive shortcut name and the raw/ folder."

Mounted at /content/drive
station_hour.csv                 found: True
stations.csv                     found: True
openmeteo_india_weather.csv      found: True


## 2. Load the three raw files

In [ ]:
station_hour = pd.read_csv(DATA_RAW / "station_hour.csv", low_memory=False)
stations = pd.read_csv(DATA_RAW / "stations.csv")
weather = pd.read_csv(DATA_RAW / "openmeteo_india_weather.csv")

RAW_ROWS = len(station_hour)
print("station_hour:", station_hour.shape)
print("stations:    ", stations.shape)
print("weather:     ", weather.shape)

# Guard rails: stop early if an input is not the one we agreed on
assert RAW_ROWS == 2_589_083, f"station_hour should have 2,589,083 rows, found {RAW_ROWS:,}"
assert list(weather.columns) == [
    "City", "Datetime", "Temperature_C", "Relative_Humidity_pct",
    "Wind_Speed_kmh", "Wind_Direction_deg",
], f"Weather columns are {list(weather.columns)}. Re-run the 03 notebook with wind direction."

station_hour: (2589083, 16)
stations:     (230, 5)
weather:      (708744, 6)


## 3. Clean the merge keys

In [ ]:
station_hour["Datetime"] = pd.to_datetime(station_hour["Datetime"])
weather["Datetime"] = pd.to_datetime(weather["Datetime"])

stations["City"] = stations["City"].astype("string").str.strip()
weather["City"] = weather["City"].astype("string").str.strip()

# AQI_Bucket has mixed types in the raw CSV; make it a clean text column
station_hour["AQI_Bucket"] = station_hour["AQI_Bucket"].astype("string")

## 4. Merge: stations, then weather (neither merge may add or drop rows)

In [ ]:
df = station_hour.merge(stations, on="StationId", how="left", validate="many_to_one")
df = df.merge(weather, on=["City", "Datetime"], how="left", validate="many_to_one")

print("After merges:", df.shape)
assert len(df) == RAW_ROWS, "A merge changed the row count"

After merges: (2589083, 24)


## 5. Fix the column order and sort

In [ ]:
D1_COLUMNS = [
    "StationId", "StationName", "City", "State", "Status", "Datetime",
    "PM2.5", "PM10", "NO", "NO2", "NOx", "NH3", "CO", "SO2", "O3",
    "Benzene", "Toluene", "Xylene",
    "AQI", "AQI_Bucket",
    "Temperature_C", "Relative_Humidity_pct", "Wind_Speed_kmh", "Wind_Direction_deg",
]

df = df[D1_COLUMNS]   # raises KeyError if any agreed column is missing

for col in ["StationId", "StationName", "City", "State", "Status"]:
    df[col] = df[col].astype("string")

df = df.sort_values(["StationId", "Datetime"]).reset_index(drop=True)
print(df.shape)
print(df.dtypes)

(2589083, 24)
StationId                string[python]
StationName              string[python]
City                     string[python]
State                    string[python]
Status                   string[python]
Datetime                 datetime64[ns]
PM2.5                           float64
PM10                            float64
NO                              float64
NO2                             float64
NOx                             float64
NH3                             float64
CO                              float64
SO2                             float64
O3                              float64
Benzene                         float64
Toluene                         float64
Xylene                          float64
AQI                             float64
AQI_Bucket               string[python]
Temperature_C                   float64
Relative_Humidity_pct             int64
Wind_Speed_kmh                  float64
Wind_Direction_deg                int64
dtype: object


## 6. Acceptance checks (from the task card)

In [ ]:
weather_cols = ["Temperature_C", "Relative_Humidity_pct", "Wind_Speed_kmh", "Wind_Direction_deg"]

# 1. Row count equals raw station_hour
assert len(df) == RAW_ROWS == 2_589_083
print("Rows:", f"{len(df):,}", "(same as raw)")

# 2. No duplicate keys
dups = df.duplicated(["StationId", "Datetime"]).sum()
assert dups == 0, f"{dups} duplicate (StationId, Datetime) pairs"
print("Duplicate (StationId, Datetime) pairs:", dups)

# 3. 24 columns, agreed order
assert list(df.columns) == D1_COLUMNS and len(df.columns) == 24
print("Columns:", len(df.columns))

# 4. Datetime is a real datetime
assert pd.api.types.is_datetime64_any_dtype(df["Datetime"])
print("Datetime dtype:", df["Datetime"].dtype)

# 5. Station metadata and weather coverage
print("\nStation metadata missing (rows):")
print(df[["StationName", "City", "State"]].isna().sum())

print("\nWeather missing (%):")
print((df[weather_cols].isna().mean() * 100).round(3))
assert df[weather_cols].isna().sum().sum() == 0, \
    "Some rows have no weather. Check the City names and the weather date ranges."

print("\nWind direction range:", df["Wind_Direction_deg"].min(), "to", df["Wind_Direction_deg"].max())
print("Date range:", df["Datetime"].min(), "to", df["Datetime"].max())
print("Stations:", df["StationId"].nunique(), "| Cities:", df["City"].nunique())
print("\nAll checks passed.")

Rows: 2,589,083 (same as raw)
Duplicate (StationId, Datetime) pairs: 0
Columns: 24
Datetime dtype: datetime64[ns]

Station metadata missing (rows):
StationName    0
City           0
State          0
dtype: int64

Weather missing (%):
Temperature_C            0.0
Relative_Humidity_pct    0.0
Wind_Speed_kmh           0.0
Wind_Direction_deg       0.0
dtype: float64

Wind direction range: 1 to 360
Date range: 2015-01-01 01:00:00 to 2020-07-01 00:00:00
Stations: 110 | Cities: 26

All checks passed.


## 7. Save D1 and the KL007 sample

In [ ]:
d1_path = DATA_PROCESSED / "station_hour_merged.parquet"
df.to_parquet(d1_path, index=False)

sample = df[df["StationId"] == "KL007"]
assert len(sample) > 0, "KL007 not found"
sample_path = DATA_SAMPLES / "KL007_sample.csv"
sample.to_csv(sample_path, index=False)

print("Saved D1:     ", d1_path, f"({d1_path.stat().st_size / 1e6:.0f} MB)")
print("Saved sample: ", sample_path, f"({len(sample):,} rows)")

Saved D1:      /content/drive/MyDrive/AQI_Project_Data/processed/station_hour_merged.parquet (58 MB)
Saved sample:  /content/drive/MyDrive/AQI_Project_Data/samples/KL007_sample.csv (3,320 rows)


## 8. Read it back (what Gokul will see)

In [ ]:
check = pd.read_parquet(d1_path)
assert check.shape == (2_589_083, 24)
assert list(check.columns) == D1_COLUMNS
assert pd.api.types.is_datetime64_any_dtype(check["Datetime"])
print("Round trip OK:", check.shape)
check.head()

Round trip OK: (2589083, 24)


,StationId,StationName,City,State,Status,Datetime,PM2.5,PM10,NO,NO2,...,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket,Temperature_C,Relative_Humidity_pct,Wind_Speed_kmh,Wind_Direction_deg
0,AP001,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,Active,2017-11-24 17:00:00,60.50,98.00,2.35,30.80,...,126.40,0.1,6.10,0.10,NaN,<NA>,31.7,46,3.5,114
1,AP001,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,Active,2017-11-24 18:00:00,65.50,111.25,2.70,24.20,...,117.12,0.1,6.25,0.15,NaN,<NA>,28.8,63,13.4,117
2,AP001,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,Active,2017-11-24 19:00:00,80.00,132.00,2.10,25.18,...,98.98,0.2,5.98,0.18,NaN,<NA>,26.7,78,10.5,149
3,AP001,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,Active,2017-11-24 20:00:00,81.50,133.25,1.95,16.25,...,112.20,0.2,6.72,0.10,NaN,<NA>,26.2,81,9.5,127
4,AP001,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,Active,2017-11-24 21:00:00,75.25,116.00,1.43,17.48,...,106.35,0.2,5.75,0.08,NaN,<NA>,25.6,84,7.1,120
